# Selective replication on GCP
Six smoke observations first, then a repeated matrix with actual worker-VM stops. The head/NFS survive. This tests our Ray scheduling and checkpoint/replication code, not TorchFT or head-loss recovery. [Details](selective_replication/gcp/README.md). Keep this notebook host awake; run cells individually.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "gossip_benchmarks").is_dir())
sys.path.insert(0, str(ROOT / "gossip_benchmarks/selective_replication"))
from gcp.launcher import GCPStudy  # noqa: E402
from gcp.suite import config  # noqa: E402

# Reuse the saved cluster. This notebook never creates/deletes VMs.
STATE_FILE = ROOT / "ray_gcp_results/rayexp-1013-194805-c4ad9f-state.json"
# If your old notebook was launched from gossip_benchmarks, adjust STATE_FILE accordingly.
assert STATE_FILE.is_file(), STATE_FILE
cloud = GCPStudy(STATE_FILE, ROOT / "ray_gcp_results/selective-vm")
print(json.dumps(cloud.inventory(), indent=2))  # Includes older labeled clusters.

In [ ]:
# Requires the original worker harness layout and all five recorded VMs RUNNING.
# Checks ownership, active suites, shared NFS, source/packages and the guest interpreter.
# Install scikit-learn==1.7.2 in guest ray-dev only if sklearn is missing.
INSTALL_MISSING_DEPENDENCIES = True
cloud.prepare(install_missing_dependencies=INSTALL_MISSING_DEPENDENCIES)

The smoke uses explicit placements to exercise both vulnerable VMs. A failed evidence gate is a reason to investigate. It is not evidence that replication is useless. Suite names are immutable: choose new names when settings change.

In [ ]:
SMOKE = "selective-vm-smoke-01"
SMOKE_CONFIG = config(epochs=1200, checkpoint_every=100, profile_repeats=1)
SMOKE_CONFIG["keep_all_checkpoints"] = False  # Keep final/restored/latest states; hash then remove unused intermediate states.
# Normalized costs, NOT current GCP prices. The overhead includes head + idle w4.
SMOKE_CONFIG["accounting"] = {"bill_warm_pool": True, "shared_storage_per_hour": 2.0}
smoke_search = cloud.prepare_suite(SMOKE, SMOKE_CONFIG)
smoke_jobs = cloud.preview(SMOKE, smoke=True)
print(f"Smoke: {len(smoke_jobs)} observations")

In [ ]:
smoke_summary = cloud.run_suite(SMOKE, smoke_jobs)
print(json.dumps(smoke_summary["checks"], indent=2))
assert smoke_summary["valid"], "Inspect smoke evidence before the full test."
print(json.dumps(smoke_summary["paired_comparisons"], indent=2))

Full test: five policy families; price × risk × deadline settings; healthy, early/late single and correlated loss; held-out fault times; five repetitions. This can run for many hours or days. Preview the actual count before executing the run cell. A passing correctness gate does not require a speedup.

In [ ]:
FULL = "selective-vm-full-01"
FULL_CONFIG = config(epochs=2400, checkpoint_every=200, profile_repeats=2)
FULL_CONFIG["keep_all_checkpoints"] = False
FULL_CONFIG["spot_cost_multipliers"] = [0.25, 1.0]
FULL_CONFIG["failure_probabilities"] = [0.1, 0.5]
FULL_CONFIG["deadline_multipliers"] = [1.25, 2.0]
FULL_CONFIG["fault_time_fractions"] = [0.2, 0.7]
FULL_CONFIG["accounting"] = {"bill_warm_pool": True, "shared_storage_per_hour": 2.0}
REPEATS = 5
CASES = None  # All 8 cases. Use an explicit list only to narrow the experiment.
assert smoke_summary["valid"]
full_search = cloud.prepare_suite(FULL, FULL_CONFIG)
full_jobs = cloud.preview(FULL, repeats=REPEATS, case_indices=CASES)
MAX_OBSERVATIONS = 1800  # Editable execution-size guard.
assert len(full_jobs) <= MAX_OBSERVATIONS
print(f"Full: {len(full_jobs)} observations")

In [ ]:
full_summary = cloud.run_suite(FULL, full_jobs, smoke_gate=SMOKE)
print(json.dumps(full_summary["checks"], indent=2))
print(json.dumps(full_summary["paired_comparisons"], indent=2))

Checkpoint sensitivity: repeat with less frequent checkpoints. Keep settings fixed before inspecting outcomes. Results remain separate.

In [ ]:
import copy

SPARSE = "selective-vm-sparse-checkpoints-01"
SPARSE_CONFIG = copy.deepcopy(FULL_CONFIG)
for task in SPARSE_CONFIG["tasks"].values():
    task["checkpoint_every"] = 800
# Run this block after the full suite when ready for the sensitivity experiment.
RUN_CHECKPOINT_SENSITIVITY = False
if RUN_CHECKPOINT_SENSITIVITY:
    cloud.prepare_suite(SPARSE, SPARSE_CONFIG)
    sparse_jobs = cloud.preview(SPARSE, repeats=REPEATS, case_indices=CASES)
    assert len(sparse_jobs) <= MAX_OBSERVATIONS
    sparse_summary = cloud.run_suite(SPARSE, sparse_jobs, smoke_gate=SMOKE)
    print(json.dumps(sparse_summary["checks"], indent=2))

In [ ]:
# Small analysis files are already downloaded after each observation.
RESULTS = cloud.local / FULL
print(RESULTS)
print("Send: manifest.json, profile.json, search.json, jobs.json, replay.json, summary.json")
# Optional full download includes all checkpoints; it may be large.
# evidence = cloud.download(FULL)
# Failed workers stay stopped after the last fault. No resources are deleted.
# Use your existing launcher to stop/clean up billable resources when finished.